In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py as hf

from dataclasses import dataclass

In [ ]:
@dataclass
class Experiment:
    loss: np.ndarray
    loss_err: np.ndarray
    entropy: np.ndarray
    entropy_err: np.ndarray
    trace: np.ndarray   
    trace_err: np.ndarray
    accuracy: np.ndarray
    accuracy_err: np.ndarray

In [ ]:
def load_data(experiment):
    """
    Load the data for a specific experiment
    """
    loss = []
    entropy = []
    trace = []
    accuracy = []

    for i in range(1, 21):
        try:
            with hf.File(f"{experiment}/train_recorder_{i}.h5", "r") as f:
                loss.append(f["loss"][:500])
                entropy.append(f["entropy"][:500])
                trace.append(f["trace"][:500])
                accuracy.append(f["accuracy"][:500])

        except FileNotFoundError:
            pass

    return Experiment(
        loss=np.nanmean(loss, axis=0),
        loss_err=np.nanstd(loss, axis=0),
        entropy=np.nanmean(entropy, axis=0),
        entropy_err=np.nanstd(entropy, axis=0),
        trace=np.nanmean(trace, axis=0),
        trace_err=np.nanstd(trace, axis=0),
        accuracy=np.nanmean(accuracy, axis=0),
        accuracy_err=np.nanstd(accuracy, axis=0),        
    )

In [ ]:
experiments = [
    "two-layer-100", "two-layer-10", "one-layer-1000", "two-layer-1000", "one-layer-100"
]

In [ ]:
experiments = {
    experiment: load_data(experiment) for experiment in experiments
    }

# Plots

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
titles = ["Loss", "Entropy", "Trace", "Accuracy"]

i = 0
epochs = np.linspace(1, 500, 500)
for key, value in experiments.items():
    ax[i].plot(value.loss, label=key)
    ax[i].fill_between(
        epochs, value.loss - value.loss_err, value.loss + value.loss_err, alpha=0.3
        )
    ax[i].set_title(titles[i])
    ax[i].set_xlabel("Epochs")
    ax[i].set_ylabel("Loss")
    ax[i].set_yscale("log")
    # ax[i].set_xscale("log")

i = 1
for key, value in experiments.items():
    ax[i].plot(value.entropy)
    ax[i].fill_between(
        epochs, value.entropy - value.entropy_err, value.entropy + value.entropy_err, alpha=0.3
        )
    ax[i].set_title(titles[i])
    ax[i].set_xlabel("Epochs")
    ax[i].set_ylabel("Entropy")
    # ax[i].set_xscale("log")

i = 2
for key, value in experiments.items():
    ax[i].plot(value.trace)
    ax[i].fill_between(
        epochs, value.trace - value.trace_err, value.trace + value.trace_err, alpha=0.3
        )
    ax[i].set_title(titles[i])
    ax[i].set_xlabel("Epochs")
    ax[i].set_ylabel("Trace")
    ax[i].set_yscale("log")
    # ax[i].set_xscale("log")

i = 3
for key, value in experiments.items():
    ax[i].plot(value.accuracy)
    ax[i].fill_between(
        epochs, value.accuracy - value.accuracy_err, value.accuracy + value.accuracy_err, alpha=0.3
        )
    ax[i].set_title(titles[i])
    ax[i].set_xlabel("Epochs")
    ax[i].set_ylabel("Trace")
    # ax[i].set_xscale("log")

fig.legend(loc=(0.4, 0.6))
plt.tight_layout()
plt.savefig("mnist_mse_analysis.png", dpi=800)
plt.show()